In [1]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
from tqdm import tqdm

/home/shkaf2m/Desktop/ml-isp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
])
val_dataset = Imagenette(root = './data', split = 'val', download = True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False, num_workers = 4)

In [4]:
def AccuracyTest(model, promts, dataset):
  model = model.to(DEVICE)
  model.eval()

  if torch.cuda.is_available():
    model = model.half()

  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in tqdm(dataset):
      images = images.to(DEVICE)
      pil_images = [torchvision.transforms.ToPILImage()(image) for image in images]
      inputs = processor(text = promts, images = pil_images, return_tensors = "pt", padding = True).to(DEVICE)
  
      outputs = model(**inputs)
      predicted = outputs.logits_per_image.argmax(dim=-1)
      total += len(images)
      correct += (predicted == labels).sum().item()
  accuracy = (correct / total)
  print(f"Accuracy Score: ", accuracy)
  print(f"Correct / Total: {correct} / {total}")

Попробуем несколько подходов:
1) Используем названия классов
2) Используем "a photo of {название класса}"
3) Для каждого класса подберем подходящие слова

In [ ]:
promts = [label[0] for label in val_dataset.classes]
print("Promts: ", promts)
AccuracyTest(model = model, promts = promts, dataset = val_dataset)

Promts:  ['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute']


100%|██████████| 3925/3925 [08:20<00:00,  7.84it/s]

Accuracy Score:  0.9677282377919321
Correct / Total: 11395 / 11775


In [6]:
promts = [f"a photo of {label[0]}" for label in val_dataset.classes]
print("Promts: ", promts)
AccuracyTest(model = model, promts = promts, dataset = val_dataset)

Promts:  ['a photo of tench', 'a photo of English springer', 'a photo of cassette player', 'a photo of chain saw', 'a photo of church', 'a photo of French horn', 'a photo of garbage truck', 'a photo of gas pump', 'a photo of golf ball', 'a photo of parachute']


  0%|          | 0/3925 [00:00<?, ?it/s]

100%|██████████| 3925/3925 [10:10<00:00,  6.43it/s]

Accuracy Score:  0.9607643312101911
Correct / Total: 11313 / 11775


In [7]:
promts = ["slimy tench", "fast English springer", "good cassette player", "electric chain saw", "religious church", "French horn", "garbage truck", "gas pump", "white golf ball", "chute parachute"]
print("Promts: ", promts)
AccuracyTest(model = model, promts = promts, dataset = val_dataset)

Promts:  ['slimy tench', 'fast English springer', 'good cassette player', 'electric chain saw', 'religious church', 'French horn', 'garbage truck', 'gas pump', 'white golf ball', 'chute parachute']


100%|██████████| 3925/3925 [09:15<00:00,  7.06it/s]

Accuracy Score:  0.9626326963906582
Correct / Total: 11335 / 11775


Теперь, попробуем усреднить эмбендинги текстовых промтов

In [12]:
model = model.to(DEVICE)
model.eval()

promts_1 = [label[0] for label in val_dataset.classes]
promts_2 = [f"a photo of {label[0]}" for label in val_dataset.classes]
promts_3 = ["slimy tench", "fast English springer", "good cassette player", "electric chain saw", "religious church", "French horn", "garbage truck", "gas pump", "white golf ball", "chute parachute"]

if torch.cuda.is_available():
  model = model.half()

correct = 0
total = 0
with torch.no_grad():
  for images, labels in tqdm(val_dataset):
    images = images.to(DEVICE)
    pil_images = [torchvision.transforms.ToPILImage()(image) for image in images]
    # texts = [promts_1]
    inputs_1 = processor(text = promts_1, images = pil_images, return_tensors = "pt", padding = True).to(DEVICE)
    inputs_2 = processor(text = promts_2, images = pil_images, return_tensors = "pt", padding = True).to(DEVICE)
    inputs_3 = processor(text = promts_2, images = pil_images, return_tensors = "pt", padding = True).to(DEVICE)
    
    outputs_1 = model(**inputs_1)
    outputs_2 = model(**inputs_1)
    outputs_3 = model(**inputs_1)
    predicted = (outputs_1.logits_per_image + outputs_2.logits_per_image + outputs_3.logits_per_image).argmax(dim = -1)
    total += len(images)
    correct += (predicted == labels).sum().item()
accuracy = (correct / total)
print(f"Accuracy Score: ", accuracy)
print(f"Correct / Total: {correct} / {total}")

  0%|          | 0/3925 [00:00<?, ?it/s]

100%|██████████| 3925/3925 [23:36<00:00,  2.77it/s]

Accuracy Score:  0.9677282377919321
Correct / Total: 11395 / 11775
